# Урок 12 - Сокращение истории чата с помощью блокнота агента

Этот ноутбук демонстрирует, как управлять контекстом в длинных разговорах с использованием Microsoft Agent Framework. По мере роста разговоров количество токенов увеличивается — в конечном итоге превышая окно контекста модели. Мы решаем эту проблему с помощью **шаблона суммирования контекста** и **блокнота агента** для постоянной памяти.

## Чему вы научитесь:
1. **Почему управление контекстом важно**: понимание лимитов токенов и окон контекста
2. **Агенты, учитывающие контекст**: создание агентов, управляющих собственным контекстом разговора
3. **Шаблон суммирования контекста**: использование инструментов для сокращения истории разговора
4. **Блокнот агента**: постоянная память, которая сохраняется при сокращении контекста

## Требования:
- Настройка Azure OpenAI с конфигурацией переменных окружения
- Понимание базовых концепций агентов из предыдущих уроков


## Установка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Почему управление контекстом важно

Каждая LLM имеет конечное **окно контекста** — максимальное количество токенов, которое она может обработать за один запрос. По мере развития многократного диалога:

- **Количество токенов растёт линейно** с каждым сообщением пользователя и ответом ассистента.
- **Токены в подсказке определяют стоимость**, потому что вся история отправляется заново на каждом шаге.
- В конечном итоге разговор **превышает окно контекста**, и модель либо усечёт данные, либо выдаст ошибку.

### Стратегии управления контекстом

| Стратегия | Как работает | Компромисс |
|---|---|---|
| **Усечение** | Отбрасываются самые ранние сообщения | Теряется ранний контекст |
| **Резюмирование** | Старые сообщения сокращаются в сводку | Часть деталей теряется, но ключевые моменты сохраняются |
| **Черновик / Внешняя память** | Ключевые факты хранятся вне разговора | Требуется вызов инструментов, но обеспечивает сохранение при любом сокращении |

В этой тетрадке мы комбинируем **резюмирование** с **инструментом черновика**, чтобы агент мог поддерживать непрерывность, даже когда история диалога сокращается.


## Создание агента с учетом контекста


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Моделирование длительного разговора

Давайте пройдемся по многоступенчатому разговору, чтобы увидеть, как накапливается контекст. Агент должен сохранять ключевые детали (предпочтения, бюджет, даты поездки) на протяжении всего разговора и демонстрировать последовательность.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обратите внимание, как агент сохраняет контекст из предыдущих реплик — он знает о Японии, суши, храмах, фотографии, бюджете в $3000, самостоятельных путешествиях и апрельском времени поездки. В короткой беседе это работает хорошо, но по мере роста диалога полная история становится дорогой для повторной отправки.

Давайте продолжим разговор с большим количеством реплик, чтобы увидеть накопление контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Паттерн Суммирования Контекста

По мере роста беседы мы можем использовать **инструмент суммирования**, чтобы сжать накопленный контекст в компактный формат. Агент вызывает этот инструмент для записи ключевых предпочтений, чтобы даже если старые сообщения будут удалены, важная информация сохранилась.

Этот паттерн является строительным блоком для более сложного сокращения истории:
1. Агент определяет ключевые факты из разговора
2. Вызывает инструмент суммирования для их сохранения
3. Старые сообщения можно безопасно удалить, поскольку сводка фиксирует важное

Ниже мы определяем инструмент `summarize_preferences`, который агент может вызвать для записи компактного резюме того, что он узнал.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резюме

В этом уроке вы узнали, как управлять контекстом в длительных беседах с агентом с использованием Microsoft Agent Framework:

### Ключевые понятия
- **Окна контекста имеют ограничение** — каждый токен в истории беседы стоит денег и учитывается в лимите.
- **Инструменты суммирования** позволяют агенту сжимать накопленный контекст в компактные сводки, уменьшая использование токенов при сохранении важной информации.
- **Записные книжки агента** обеспечивают постоянную внешнюю память, которая сохраняется при любом сокращении беседы.

### Что вы создали
- **Агента, учитывающего контекст**, который поддерживает непрерывность в многоходовых беседах
- **Инструмент суммирования** (`summarize_preferences`), который записывает ключевые детали пользователя в компактном формате
- **Многоходовую беседу**, демонстрирующую сохранение и изменение контекста

### Реальные применения
- **Боты службы поддержки**: запоминают предпочтения в ходе длительных сеансов поддержки
- **Личные помощники**: отслеживают текущие проекты без необходимости повторного объяснения контекста
- **Образовательные репетиторы**: сохраняют прогресс ученика в течение множества взаимодействий

### Следующие шаги
- Реализовать полноценный инструмент записной книжки с файловым хранением данных
- Добавить автоматическое усечение истории после суммирования
- Объединить с векторными базами данных для семантического поиска памяти
- Создать агентов, которые могут возобновлять беседы спустя дни, имея полный контекст


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с использованием автоматического переводческого сервиса [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, пожалуйста, учитывайте, что машинный перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для критически важной информации рекомендуется профессиональный перевод, выполненный человеком. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
